# 02 - EDA on raw PeMS data: I-5, District 12 (Orange County) and District 7 (Los Angeles), 2023-2025

**Data:** 5-minute PeMS station data for the I-5 corridor, mainline lanes only, 2023-2025.

**Questions**
1. What is in the files?
2. How much of the data is a PeMS estimate rather than a real measurement, and where and when?
3. What does traffic flow look like along the corridor?
4. Where and when is the road congested?

## 0. Setup and load

The daily files are combined once into one file per district (mainline only). If the combined file
already exists, it is just loaded.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import missingno as msno            # if missing: pip install missingno

# Find the repo folder: start where this notebook runs and go up until a folder contains "Data".
# This works wherever the notebook is saved inside the repo (notebooks/ or notebooks/<your folder>/).
print("working folder:", Path.cwd())
ROOT = Path.cwd()
while not (ROOT / "Data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "Data").is_dir():
    raise FileNotFoundError("Couldn't find the repo's Data folder above " + str(Path.cwd()) +
                            ". Open the notebook from inside the repo folder.")
print("repo folder:", ROOT)

DATA = ROOT / "Data"                       # raw PeMS files: Data/D12 2023/..., Data/D7 2025/...
PROCESSED = ROOT / "data" / "processed"    # combined files (on Windows, "data" and "Data" are the same folder)

In [ ]:
columns = ["timestamp", "station", "lane_type", "pct_observed", "flow", "occupancy", "speed"]
PROCESSED.mkdir(parents=True, exist_ok=True)

for district in ["D12", "D7"]:
    out_file = PROCESSED / f"{district}_2023_2025_mainline.parquet"
    if out_file.exists():
        print(out_file.name, "already exists")
        continue

    all_days = []
    for year in [2023, 2024, 2025]:
        folder = DATA / f"{district} {year}" / "pems_parquet" / "station_5min"
        for name in sorted(os.listdir(folder)):
            day = pd.read_parquet(folder / name, columns=columns)
            day = day[day["lane_type"] == "ML"]
            all_days.append(day)
        print(district, year, "loaded")

    combined = pd.concat(all_days, ignore_index=True)
    combined.to_parquet(out_file)
    print(district, combined.shape, "saved to", out_file)
    del all_days, combined

In [ ]:
d12 = pd.read_parquet(PROCESSED / "D12_2023_2025_mainline.parquet")
d7 = pd.read_parquet(PROCESSED / "D7_2023_2025_mainline.parquet",
                     columns=["timestamp", "station", "pct_observed"])     # only what is compared below
print("D12:", d12.shape, "| D7:", d7.shape)
d12.head()

In [ ]:
stations = pd.read_csv(DATA / "D12 2023" / "pems_parquet_stations.csv")
ml = stations[stations["Type"] == "ML"].sort_values("Abs_PM")     # mainline, south (upstream) to north
CORRIDOR_IDS = ml["ID"].tolist()
meta = ml.set_index("ID")
print(len(CORRIDOR_IDS), "mainline sensors in D12")
meta[["Name", "Abs_PM", "Lanes"]].head()

## 1. What is in the file?

**Questions:** How many days, sensors and timestamps? Are there missing values or duplicates?

In [ ]:
n_days = d12["timestamp"].dt.date.nunique()
n_timestamps = d12["timestamp"].nunique()
print("days:", n_days)
print("sensors:", d12["station"].nunique())
print("distinct timestamps:", n_timestamps)
print("expected (days x 288):", n_days * 288)
print("missing timestamps:", n_days * 288 - n_timestamps)

In [ ]:
print("empty cells per column:")
print(d12.isna().sum())
print()
print("exact duplicate rows:", d12.duplicated().sum())
print("days with duplicates:", sorted(d12.loc[d12.duplicated(), "timestamp"].dt.date.unique()))

In [ ]:
d12 = d12.drop_duplicates()
d12[["pct_observed", "flow", "occupancy", "speed"]].describe().round(2)

In [ ]:
readings_per_slot = d12.groupby("timestamp").size()
full = pd.date_range(d12["timestamp"].min(), d12["timestamp"].max(), freq="5min")
missing_slots = full.difference(readings_per_slot.index)
print(len(missing_slots), "5-minute slots with no data at all")
pd.Series(missing_slots.date).value_counts().sort_index()

**Finding:**
- **Coverage:** 1,094 days of files and 46 mainline sensors in D12. Across the full date range,
  1,190 five-minute slots have no data at all:
  - the missing hour on each spring daylight-saving morning (12 slots, 3 times)
  - part of 2023-12-11 (112 slots) and 2025-02-01 (219 slots)
  - Nov 27-29, 2025, around Thanksgiving: 247 slots, and Nov 28-29 have no files at all
- **Duplicates:** there are 5,750 exact duplicate rows, on 2023-02-02 and 2025-05-26 (early morning
  only). They are dropped above.
- **No empty cells at all.** That does *not* mean every value was measured: PeMS fills gaps with its
  own estimates. Section 2 uses `pct_observed` to tell real readings from estimates.

## 2. How much of the data is a PeMS estimate?

`pct_observed` is the share of a sensor's lanes that were actually measured in that 5-minute
reading. 100 = fully measured, 0 = fully estimated by PeMS.

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(7, 7), sharex=True)
axs[0].hist(d7["pct_observed"], bins=20, color="#256abf")
axs[0].set_title("Percentage of observed data in District 7")
axs[0].set_ylabel("Number of readings")
axs[1].hist(d12["pct_observed"], bins=20, color="#256abf")
axs[1].set_title("Percentage of observed data in District 12")
axs[1].set_xlabel("Percentage of observed data")
axs[1].set_ylabel("Number of readings")
plt.tight_layout()
plt.show()

In [ ]:
share = pd.DataFrame({
    "D12": (d12["pct_observed"] == 0).groupby(d12["timestamp"].dt.year).mean() * 100,
    "D7": (d7["pct_observed"] == 0).groupby(d7["timestamp"].dt.year).mean() * 100,
}).round(1)
share.loc["all years"] = [(d12["pct_observed"] == 0).mean() * 100, (d7["pct_observed"] == 0).mean() * 100]
print("% of readings fully estimated (pct_observed = 0)")
share.round(1)

### Which sensors can we trust?

In [ ]:
d12["obs_status"] = pd.cut(
    d12["pct_observed"],
    bins=[-1, 0, 99.9, 100],
    labels=["estimated (0%)", "partly observed", "fully observed (100%)"],
)
labels = meta["Name"] + " (" + meta["Abs_PM"].round(1).astype(str) + ")"

status = (
    d12.groupby(["station", "obs_status"], observed=False).size()
        .unstack(fill_value=0)
        .reindex(CORRIDOR_IDS)
)
status = status.div(status.sum(axis=1), axis=0) * 100      # counts -> percentages per sensor
status.index = labels.reindex(CORRIDOR_IDS)

status.plot(kind="bar", stacked=True, figsize=(16, 5), width=0.85,
            color=["#d9d9d9", "#9ec5f4", "#256abf"])
plt.title("How much of each sensor's data is real? (south / upstream on the left), 2023-2025")
plt.ylabel("% of 5-minute readings")
plt.xlabel("")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

### When are readings estimated?

In [ ]:
d12["estimated"] = d12["pct_observed"] == 0
d12["hour"] = d12["timestamp"].dt.hour
d12["weekend"] = d12["timestamp"].dt.dayofweek >= 5

(d12.groupby(["hour", "weekend"])["estimated"].mean() * 100).unstack().plot(marker="o", figsize=(8, 4))
plt.ylabel("% of readings fully estimated")
plt.xlabel("Hour of day")
plt.legend(["Weekday", "Weekend"])
plt.title("D12: estimated readings by hour")
plt.tight_layout()
plt.show()

In [ ]:
monthly = d12.groupby(d12["timestamp"].dt.to_period("M"))["estimated"].mean() * 100
monthly.plot(kind="bar", figsize=(14, 4), color="#256abf")
plt.ylabel("% of readings fully estimated")
plt.xlabel("")
plt.title("D12: estimated readings by month")
plt.tight_layout()
plt.show()

### Missingness map

The raw table has no empty cells, so `missingno` shows nothing useful on it directly:

In [ ]:
msno.bar(d12[["timestamp", "station", "pct_observed", "flow", "occupancy", "speed"]].sample(100_000, random_state=0),
         figsize=(8, 3))
plt.show()

To see where the gaps really are, estimated speeds are turned into empty values, and the data is
put in a wide table with one row per 5-minute slot and one column per sensor (south to north).
In the plot below, white means "no real reading".

In [ ]:
d12["speed_obs"] = d12["speed"].where(d12["pct_observed"] > 0)
wide = d12.pivot(index="timestamp", columns="station", values="speed_obs")
wide = wide.reindex(index=pd.date_range(wide.index.min(), wide.index.max(), freq="5min"),
                    columns=CORRIDOR_IDS)
print(wide.shape, "| share of empty cells:", round(wide.isna().mean().mean(), 3))

msno.matrix(wide, freq="QS", figsize=(16, 7), sparkline=False)
plt.show()

In [ ]:
msno.heatmap(wide, figsize=(14, 11), fontsize=8)
plt.show()

**Finding:**
- **D12 is far more usable than D7.** About 27% of D12 readings are fully estimated, against about
  82% in D7.
- **D12 improves every year:** 45% fully estimated in 2023, 22% in 2024 and 13% in 2025.
- **Summer 2023 outage:** in June-August 2023 most of D12 was estimated (70-84% per month).
- **Some sensors are mostly estimated:** Katella (87% of readings), Main 2 (82%), Euclid 1 (64%),
  N of 55 (58%) and Jamboree 1 (57%, and never fully observed). Their numbers should not be trusted.
- **Time of day hardly matters:** about 24% of readings are estimated at night and 27-28% during the
  day, with weekends slightly lower. Missing data is not a night-time problem.
- **Gaps cluster:** neighboring sensors tend to go missing together (the heatmap), often for weeks
  at a time (the matrix).
- **Decision for modeling:** use D12, real readings only, and leave out or downweight 2023.

### Do estimated speeds look like real ones?

In [ ]:
sns.histplot(data=d12.sample(500_000, random_state=0), x="speed", hue="estimated",
             bins=60, stat="density", common_norm=False, element="step")
plt.title("Speed distribution: real vs. estimated readings")
plt.xlabel("Speed (mph)")
plt.show()

**Finding:** estimated speeds pile up around 60-65 mph. They have fewer readings above 70 mph and
fewer in the 30-45 mph range, where real jams show up as a small bump. A model trained on estimates
would learn PeMS's guessing rule and see fewer jams than really happened.

## 3. What does the flow look like along the corridor?

Flow in PeMS is vehicles per 5 minutes (all lanes), so x 12 gives vehicles per hour. Real readings
only.

In [ ]:
real = d12[~d12["estimated"]]
flow_by_hour = (real.groupby(["station", "hour"])["flow"].mean() * 12).unstack().reindex(CORRIDOR_IDS)
flow_by_hour.index = labels.reindex(CORRIDOR_IDS)

plt.figure(figsize=(14, 11))
sns.heatmap(flow_by_hour, cmap="Blues", cbar_kws={"label": "vehicles per hour"})
plt.title("Average flow by sensor (south at the top) and hour of day, real readings")
plt.xlabel("Hour of day")
plt.ylabel("")
plt.tight_layout()
plt.show()

### Busiest sensors through the day (weekdays)

In [ ]:
weekday = real[~real["weekend"]]
busiest = weekday.groupby("station")["flow"].mean().sort_values(ascending=False).index[:5]
for sid in busiest:
    profile = weekday[weekday["station"] == sid].groupby("hour")["flow"].mean() * 12
    plt.plot(profile.index, profile.values, marker="o", label=labels[sid])
plt.xlabel("Hour of day")
plt.ylabel("vehicles per hour")
plt.title("Weekday flow at the 5 busiest sensors")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Flow vs. speed at one busy sensor

Free-flowing traffic sits on the upper branch: high speed, flow rising with demand. Congested
traffic falls onto the lower branch, where speed drops and flow drops with it.

In [ ]:
sid = busiest[0]
one = weekday[weekday["station"] == sid].sample(20_000, random_state=0)
plt.scatter(one["flow"] * 12, one["speed"], s=3, alpha=0.3, color="#256abf")
plt.xlabel("flow (vehicles per hour)")
plt.ylabel("speed (mph)")
plt.title(f"Flow vs speed, {labels[sid]}, weekdays, real readings")
plt.tight_layout()
plt.show()

**Finding:**
- **Peak flow is in the morning.** At the busiest sensors (Grand, B St., Concord, 17th, around
  postmile 102-105) weekday flow peaks at 7-8 am at about 8,500-9,500 vehicles per hour. It stays
  high through midday.
- **The 5 pm dip is congestion, not lower demand.** Flow drops to about 6,000-7,000 vehicles per hour
  at 5 pm, when the most cars want to travel, because the road is jammed. It recovers around 7 pm.
- **Check Grand 2:** its night-time flow is unusually high (about 5,000 vehicles per hour at 2-3 am,
  against under 1,000 at its neighbors). That looks like a detector or configuration problem.
- **The flow-speed plot shows two traffic states.** In free flow, speed is 55-75 mph and flow rises up
  to about 11,000 vehicles per hour. In congestion, speed drops to 30-50 mph and flow falls back to
  about 7,000-8,500.

## 4. Where and when is the road congested?

**Free-flow speed** = each sensor's 85th percentile of real speeds. **Congested** = speed below 60%
of it.

In [ ]:
free_flow = real.groupby("station")["speed"].quantile(0.85)
d12["congested"] = d12["speed"] < 0.6 * d12["station"].map(free_flow)

summary = (
    d12.groupby("station")
        .agg(mean_flow=("flow", "mean"),
             pct_observed=("pct_observed", "mean"),
             congested_share=("congested", "mean"))
        .reindex(CORRIDOR_IDS)
        .join(meta[["Name", "Abs_PM", "Latitude", "Longitude", "Lanes"]])
)
summary["free_flow_mph"] = free_flow
summary["vehicles_per_hour"] = summary["mean_flow"] * 12
summary["label"] = labels
summary.sort_values("congested_share", ascending=False).head(10).round(3)

In [ ]:
fig = px.scatter_map(
    summary.reset_index(names="station"),
    lat="Latitude", lon="Longitude",
    color="congested_share",
    size="vehicles_per_hour",
    hover_name="label",
    hover_data={"pct_observed": ":.0f", "vehicles_per_hour": ":.0f", "free_flow_mph": ":.0f"},
    color_continuous_scale="Blues",
    zoom=10.5, height=600,
    map_style="open-street-map",
)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))
fig.show()

In [ ]:
daily = d12.groupby(d12["timestamp"].dt.date)["congested"].mean() * 100
daily.index = pd.to_datetime(daily.index)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily.index, daily.values, color="#9ec5f4", linewidth=0.8, label="daily")
ax.plot(daily.index, daily.rolling(28, center=True).mean(), color="#256abf", linewidth=2, label="4-week average")
ax.set_title("Share of corridor congested each day")
ax.set_ylabel("% of sensor-readings congested")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = d12["timestamp"].dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col, title in [
    (axes[0], "congested", "% of time congested"),
    (axes[1], "estimated", "% of readings estimated by PeMS"),
]:
    table = d12.groupby([dow, d12["hour"]])[col].mean().unstack() * 100
    table.index = days
    sns.heatmap(table, cmap="Blues", vmin=0, ax=ax, cbar_kws={"label": "%"})
    ax.set_title(title)
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

**Finding:**
- **When:** northbound I-5 in Orange County congests on **weekday afternoons**, mostly 3-7 pm and
  worst Tuesday-Thursday around 5 pm. Mornings and weekends are mostly free-flowing in this direction.
- **Where:** the most congested sensors are in the southern half of the corridor (Tustin Ranch, Red
  Hill, 17th, Grand) and around Anaheim (Gene Autry, Orangewood). The northern end (past Euclid) is
  rarely congested.
- **Over time:** the 4-week average is about 4-7% of sensor-readings congested. The daily line swings
  every week because weekends are nearly free of congestion. There are dips around the year-end
  holidays and a long low in summer 2023 (2-3%). That low is the detector outage, when most values
  are PeMS estimates, which show less congestion than really happened.
- **Next steps:** model weekdays 6 am-8 pm, real readings only, with 2023 used carefully.